# Lab 8 - Harness Agent を workflow に組み込み、Hosted Agent にする

Lab 7 で観察した `travel_harness_agent` を、入力整理と最終レビューを担当する通常 Agent の間に置きます。3 participant の引き継ぎをコード・グラフ・途中回答で確認してから、同じ checked-in source を Microsoft Foundry の Hosted Agent として deploy します。

```text
intake_agent -> travel_harness_agent -> reviewer_agent
```

**Lab 7 の Notebook 実行結果には依存しません。** この Notebook は shared factory から新しい Harness Agent を作ります。Lab 3 の Foundry IQ と Lab 4 の Toolbox があれば、経験者は Lab 7 を飛ばして実行できます。

**使用する kernel:** `Python (Foundry Hosted Agent)`。Lab 1 で選んだ VS Code / JupyterLab で同じ Notebook を実行します。Cloud Shell では起動スクリプトが Graphviz の PATH を設定します。

> **境界と料金:** モデル利用料金に加え、Foundry IQ、Toolbox tools、source remote build、Hosted Agent の稼働には別途の実行料金が発生します。入力と途中回答は Azure へ送信されます。架空のデータだけを使ってください。Notebook の Run All ではデプロイしません。

## 1. Hosted workflow の接続先を読み込む

Lab 1 の `.workshop/context.json` から model と Search endpoint を取得し、Lab 3/4 の resource 名を設定します。`FOUNDRY_PROJECT_ENDPOINT` はローカル実行用です。deploy 後は Hosted Agent platform が自動注入します。

In [ ]:
import json
import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "hosted-agent" / "workflow.py").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。Lab 1 で準備した教材内の Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError("Lab 1 を完了し、.workshop/context.json を作成してください。")

context = json.loads(context_path.read_text(encoding="utf-8"))
outputs = context["terraform_outputs"]
os.environ["FOUNDRY_PROJECT_ENDPOINT"] = outputs["foundry_project_endpoint"]["value"]
os.environ["FOUNDRY_MODEL"] = outputs["primary_model_deployment_name"]["value"]
os.environ["AZURE_AI_SEARCH_SERVICE_ENDPOINT"] = outputs["search_service_endpoint"]["value"]
os.environ["AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME"] = "contoso-travel-knowledge-lab"
os.environ["TOOLBOX_NAME"] = "contoso-travel-toolbox"

hosted_source = REPO_ROOT / "src" / "hosted-agent"
if str(hosted_source) not in sys.path:
    sys.path.insert(0, str(hosted_source))

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model: {os.environ['FOUNDRY_MODEL']}")
print(f"Foundry IQ: {os.environ['AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME']}")
print(f"Toolbox: {os.environ['TOOLBOX_NAME']}")

## 2. 役割の異なる 3 participant を作る

| Participant | 担当 | 持つ機能 |
|---|---|---|
| `intake_agent` | 依頼の値と不足項目を整理 | model + instructions |
| `travel_harness_agent` | todo に分解し、規程・Skill・tool を使って調査と計算 | Foundry IQ + Toolbox + Skills + Tool Search + Harness |
| `reviewer_agent` | 根拠と tool 結果を検査し、最終回答に整える | model + instructions |

Lab 7 の plan mode は人が計画を確認する対話用でした。workflow の途中で止まらないよう、ここでは Harness Agent を **execute mode** で作ります。interactive な plan 承認を省いても、複雑な依頼では todos と bounded loop を使います。

In [ ]:
from contextlib import AsyncExitStack

import travel_agents
import workflow
from agent_framework import FileSystemAgentFileStore

credential = travel_agents.create_credential()
chat_client = travel_agents.create_chat_client(credential)
intake_agent = chat_client.as_agent(
    name="intake_agent",
    instructions=workflow.INTAKE_AGENT_INSTRUCTIONS,
)
travel_harness_agent = travel_agents.build_environment_harness_agent(
    default_mode="execute",
    hosted=True,
    file_memory_store=FileSystemAgentFileStore(
        str(REPO_ROOT / ".workshop" / "lab8-harness-memory")
    ),
)
reviewer_agent = chat_client.as_agent(
    name="reviewer_agent",
    instructions=workflow.REVIEWER_AGENT_INSTRUCTIONS,
)
participants = [intake_agent, travel_harness_agent, reviewer_agent]
expected_order = [agent.name for agent in participants]
print(" -> ".join(expected_order))

resource_stack = AsyncExitStack()
await resource_stack.__aenter__()
for participant in participants:
    await resource_stack.enter_async_context(participant)

## 3. `SequentialBuilder` で順番を固定する

Harness Agent 自体は 1 つの Agent の実行支援です。`SequentialBuilder` は複数 participant の実行順と会話の引き継ぎを定義します。元の依頼と、それまでの participant の回答が次へ渡ります。

`output_from=[reviewer_agent]` で最終回答を reviewer に限定し、Notebook だけ `intermediate_output_from="all_other"` を使って intake / Harness の途中回答を観察します。deploy 用 source は reviewer の最終回答だけを公開します。

In [ ]:
from agent_framework import Workflow
from agent_framework.orchestrations import SequentialBuilder


def build_travel_workflow() -> Workflow:
    return SequentialBuilder(
        participants=participants,
        output_from=[reviewer_agent],
        intermediate_output_from="all_other",
    ).build()


travel_workflow = build_travel_workflow()

## 4. 実際の workflow graph を表示する

`WorkflowViz` は今作った workflow object から graph を生成します。`intake_agent -> travel_harness_agent -> reviewer_agent` の順を確認してください。

In [ ]:
import shutil
import subprocess

from agent_framework import WorkflowViz
from IPython.display import SVG, Code, display

viz = WorkflowViz(travel_workflow)
mermaid_graph = viz.to_mermaid()
dot_executable = shutil.which("dot")
if dot_executable is None:
    print("Graphviz がないため Mermaid 定義を表示します。")
    display(Code(mermaid_graph, language="text"))
else:
    rendered = subprocess.run(
        [dot_executable, "-Tsvg"],
        input=viz.to_digraph(),
        capture_output=True,
        text=True,
        encoding="utf-8",
        check=False,
    )
    if rendered.returncode != 0:
        print(rendered.stderr)
        rendered.check_returncode()
    display(SVG(data=rendered.stdout))

## 5. 途中回答と最終回答を観察する

標準の合成依頼には規程確認、Travel Ops API の見積もり、Code Interpreter による予算比較が含まれます。予約・承認シミュレーションは明示的に不要と指定し、不要な tool が呼ばれないことも確認します。

stream event を participant ごとに分けて表示します。Harness 内部の `load_skill` / `tool_search` / `call_tool` は Lab 7 と Portal Trace で確認し、ここでは participant 間の引き継ぎに注目します。

In [ ]:
from agent_framework import AgentResponseUpdate
from IPython.display import Markdown, display

user_request = workflow.SAMPLE_REQUEST
execution_order = []
intermediate_answers = {}
harness_actions = set()
final_chunks = []
answer = None
travel_workflow = build_travel_workflow()

async for event in travel_workflow.run(user_request, stream=True):
    if event.type in {"failed", "executor_failed", "error"}:
        raise RuntimeError(f"Workflow failed: {event.details or event.data}")
    if event.executor_id == travel_harness_agent.name and isinstance(
        event.data, AgentResponseUpdate
    ):
        harness_actions.update(
            content.name for content in event.data.contents if getattr(content, "name", None)
        )
    if event.type == "executor_invoked" and event.executor_id in expected_order:
        execution_order.append(event.executor_id)
        print(f"\n開始: {event.executor_id}", flush=True)
    elif event.type == "intermediate" and isinstance(event.data, AgentResponseUpdate):
        intermediate_answers.setdefault(event.executor_id, "")
        intermediate_answers[event.executor_id] += event.data.text
        print(event.data.text, end="", flush=True)
    elif event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        final_chunks.append(event.data.text)

answer = "".join(final_chunks)
if not answer.strip():
    raise RuntimeError("reviewer の最終回答が空です。実行ログを確認してください。")
display(Markdown("### reviewer_agent の最終回答"))
print(answer)
print("Harness actions:", sorted(harness_actions))

In [ ]:
assert execution_order == expected_order, execution_order
assert set(intermediate_answers) == {"intake_agent", "travel_harness_agent"}
assert all(intermediate_answers.values())
required_actions = {"load_skill", "knowledge_base_retrieve", "tool_search", "call_tool"}
assert required_actions <= harness_actions, (
    "回答文だけでなく、必要な Skill・検索・API の実行を確認してください。",
    harness_actions,
)
assert all(heading in answer for heading in ["規程確認", "概算", "次のアクション"])
assert workflow.SIMULATION_NOTICE in answer
print("OK: participant 順序、途中回答、reviewer の最終形式を確認しました。")

## 6. Notebook と deploy source の対応を確認する

Notebook は学習のため participant 作成を展開しました。deploy 対象は `src/hosted-agent/` です。次の source が同じ shared Harness factory と participant 順を使うことを確認します。

In [ ]:
import inspect

display(Code(inspect.getsource(workflow.build_workflow), language="python"))
display(Code(inspect.getsource(travel_agents.build_harness_travel_agent), language="python"))

## 7. Azure を使わない contract test を実行する

実モデルの回答は変動します。fake client を使うテストでは、participant 順、会話の引き継ぎ、Harness Agent が `SequentialBuilder` の participant として動くこと、reviewer だけが最終回答になることを固定して確認します。

In [ ]:
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/contract/hosted_agent/test_sequential_workflow.py",
        "tests/contract/hosted_agent/test_telemetry.py",
        "-q",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    check=False,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    completed.check_returncode()

## 8. ローカル接続を閉じる

MCP session と HTTP client を閉じます。Azure resource は削除しません。

In [ ]:
await resource_stack.aclose()
credential.close()
print("Local workflow resources closed.")

## 9. Terminal から Hosted Agent を deploy する

repository root の Terminal で、Notebook kernel とは別の root `.venv` を使います。

```bash
.venv/bin/python scripts/deploy_hosted_agent.py --output json
```

script は source を package し、model / Search / knowledge base / Toolbox の環境値を設定し、Python 3.13 remote build を開始します。作成された agent identity には Search Index Data Reader、Toolbox Skills に必要な Foundry User、trace 送信用の Monitoring Metrics Publisher を resource scope で冪等に付与します。

`status: "active"` を確認したら [Lab 8 の Portal 手順](../labs/08-hosted-multi-agent.md#3-portal-で実行する)へ戻ります。Trace と cleanup は [Lab 9](../labs/09-observability-cleanup.md) で行います。